In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [ ]:
llm = ChatOpenAI()

def chat_node(state:ChatState):
    messages = state['messages']
    response = llm.invoke(messages)

    return {'messages':[response]}

In [ ]:
graph = StateGraph(ChatState)
checkpoint = MemorySaver()

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

workflow = graph.compile(checkpointer=checkpoint)

In [ ]:
thread_id = '1'

while True:
    user_message = input('Type here: ')
    print('User',user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    response = workflow.invoke({'message': HumanMessage(content=user_message)},config=config)

    print('AI: ',response['messages'][-1].content)